# 📘 Scripting Best Practices, PEP8 & FastAPI Introduction

A self-study notebook. Read each section top to bottom, run every code cell, and try the "Try It Yourself" prompts before moving on.

**What you'll learn:**
1. Part A — Scripting Best Practices (how to write scripts like a professional, not a beginner)
2. Part B — PEP8 (Python's official style guide)
3. Part C — FastAPI Introduction (what APIs are, how the web talks to itself, and why FastAPI)


---
# Part A — Scripting Best Practices


## A0. What is a Script?

A **script** is a small program written to automate a specific task — not a full application with a UI, users, or a database behind it (usually). Think of things like:

- A file renamer that cleans up 500 messy filenames in one run
- A log cleaner that deletes logs older than 30 days
- A CSV-to-report converter that turns raw exported data into a summary
- A backup automation script that copies files to another folder every night
- An API health-check script that pings a server every 5 minutes and alerts if it's down

Scripts are usually run from the command line, often on a schedule (cron jobs, Windows Task Scheduler), and often written by one person but used by an entire team.


## A1. The Hidden Goal of a Good Script

Most beginners think the goal of a script is simple: **"it works when I run it."** That's not enough in the real world.

The *real* goal of a good script is:

- **Reusability** — someone else (or future-you) should be able to run it without editing the code first
- **Readability** — anyone on the team should be able to open it and understand what it does within a minute
- **Safety** — it shouldn't silently corrupt data, crash halfway through a job, or need babysitting

A script you write today might get scheduled to run every night for the next three years, or get inherited by the next batch of interns. Treat it like production code, because eventually, it becomes production code.


## A2. Professional Script Structure

**Beginner style** — everything dumped top-to-bottom with no structure:


In [1]:
# Beginner style (avoid this)
print("Start")
data = [1, 2, 3]
total = sum(data)
print(total)


Start
6


**Professional style** — organized into clear sections: imports, constants, functions, and a `main()` entry point:


In [2]:
# Professional structure
import logging

DEFAULT_LIMIT = 100

def process(data):
    return sum(data)

def main():
    result = process([1, 2, 3])
    print(result)

if __name__ == "__main__":
    main()


6


Notice the pattern: **imports** at the top → **constants** next → **functions** (the actual logic) → a **`main()`** function that ties it together → a guard at the bottom that calls `main()`. Almost every real-world Python script, tool, and CLI utility follows this exact skeleton.


## A3. Why Do We Use `if __name__ == "__main__":`?

Every Python file has a hidden variable called `__name__`. When you *run* a file directly, Python sets `__name__` to `"__main__"`. When you *import* that same file into another file, `__name__` is set to the module's name instead.

This guard means: **"only run this code when the file is executed directly — not when it's imported."**


In [3]:
if __name__ == "__main__":
    main()


6


Without this guard, the moment anyone writes `import your_script`, your entire script would execute immediately — including any test code, print statements, or side effects you didn't intend to trigger. This is exactly why unit test files can safely `import` your functions without accidentally running your whole script.


## A4. Single Responsibility Principle (SRP)

**SRP says: one function should do one job.** If a function's name needs the word "and" to describe it ("read file *and* validate *and* calculate *and* save"), it's doing too much.

**Bad — one giant function does everything:**


In [4]:
# Bad — one function does everything
def process_all(path):
    data = open(path).read()
    # validate + calculate + print + save all mixed together here...


**Good — each responsibility gets its own function:**


In [5]:
# Good — split responsibilities
def read_file(path):
    ...

def validate(data):
    ...

def calculate(data):
    ...

def save_report(result):
    ...


Why this matters: if each function does one thing, you can **test it in isolation**, **reuse it elsewhere**, and **fix a bug in one place** without fear of breaking three other things. This is exactly why large, testable codebases (any real backend service) are built out of many small functions rather than a few giant ones.


## A5. Naming Matters

Names like `x`, `d`, `data1`, `temp` tell the reader nothing. A good variable name should describe **what it holds** or **what it does** — not how quickly you typed it.


In [6]:
# Bad
d = 100
x = d * 1.18

# Good
invoice_data = 100
invoice_with_tax = invoice_data * 1.18


In real code review at any company, naming is consistently the single most commented-on issue — more than logic bugs. Clear names reduce the time it takes a new teammate to understand your code from minutes to seconds.


## A6. Constants

**Constants** are fixed values that don't change while the program runs. By convention, they're written in `UPPER_CASE` and declared near the top of the file, not scattered inline through the logic.


In [7]:
MAX_RETRIES = 3
TAX_RATE = 0.18
API_TIMEOUT_SEC = 30


Config-driven systems — feature flags, rate limits, retry counts, timeout thresholds — all start life as a named constant like this. It's much easier to tune one value at the top of a file than to hunt for a "magic number" buried somewhere in the logic.


## A7. Avoid Hardcoding

**Hardcoding** means embedding a value directly into your logic instead of pulling it from a constant, config file, or environment variable. The problem: when that value needs to change, you have to hunt through the code to find every place it appears.


```python
# Bad — magic number buried in logic
if attempts > 3:
    raise Exception("Too many attempts")
```


In [8]:
# Good
MAX_ATTEMPTS = 3
attempts = 5
try:
    if attempts > MAX_ATTEMPTS:
        raise Exception("Too many attempts")
except Exception as e:
    print("Caught:", e)


Caught: Too many attempts


Hardcoded database passwords, URLs, and API keys are a classic real-world bug *and* security risk — they should always live in environment variables or config files (e.g. a `.env` file), never directly in source code.


## A8. DRY — Don't Repeat Yourself

If the same logic is copy-pasted in three places, a bug fix later means you have to remember to fix it in all three places — and it's easy to forget one.


In [9]:
# Bad — repeated formula
price1 = 100 * 1.18
price2 = 200 * 1.18
price3 = 300 * 1.18

# Good — extracted into a reusable function
def with_tax(price, rate=0.18):
    return price * (1 + rate)

print(with_tax(100), with_tax(200), with_tax(300))


118.0 236.0 354.0


"Fixed the bug in one place, forgot the other copy" is one of the most common real production incident write-ups. DRY code means there's only ever one place to fix.


## A9. Handle Errors Properly

A script should never crash and dump a raw, scary traceback in front of a non-technical user. Wrap risky operations in `try`/`except`, and handle the specific errors you expect.

**Bad — crashes ugly on bad input:**


```python
value = int(input("Enter number: "))
print(10 / value)
```


If the user types `"abc"`, this crashes with a `ValueError`. If they type `"0"`, it crashes with a `ZeroDivisionError`. Neither is a graceful experience.

**Good — handled:**


In [10]:
def safe_divide(raw_value):
    try:
        value = int(raw_value)
        return 10 / value
    except ValueError:
        print("Please enter a valid number.")
    except ZeroDivisionError:
        print("Cannot divide by zero.")

print(safe_divide("abc"))
print(safe_divide("0"))
print(safe_divide("5"))


Please enter a valid number.
None
Cannot divide by zero.
None
2.0


Every real API you've used (FastAPI, Flask, Django) wraps business logic in structured error handling so it can return a clean, meaningful HTTP error (like `400 Bad Request`) instead of crashing the whole server.


## A10. Logging Instead of `print()`

`print()` is fine for a quick sanity check while you're writing code — but the moment the terminal closes, that output is gone forever. `logging` gives you timestamps, severity levels (INFO/WARNING/ERROR), and the ability to write to a file or send to a monitoring system.


In [11]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

logger.info("Script started")
logger.warning("Low disk space")
logger.error("Failed to connect to DB")


INFO:__main__:Script started


ERROR:__main__:Failed to connect to DB


Production systems ship their logs to centralized tools like the ELK stack, Datadog, or CloudWatch, where engineers search and alert on them. A stray `print()` statement is invisible to all of that — nobody is watching a terminal that isn't open.


## A11. Use Virtual Environments

A **virtual environment** creates an isolated Python installation just for your project, so its dependencies don't clash with another project's dependencies on the same machine.


```bash
python -m venv venv
source venv/bin/activate   # venv\Scripts\activate on Windows
pip install -r requirements.txt
```


Without this, you can run into the classic "works on my machine" problem — Project A needs `requests==2.25`, Project B needs `requests==2.31`, and installing one breaks the other if they share one global Python environment. Every real project (and every CI/CD pipeline) creates a fresh, isolated environment and installs pinned dependency versions into it.


## A12. Format Your Code (a preview of PEP8)

Consistent formatting means any teammate can read your code instantly, without adjusting to your personal style. Tools exist to automate this so you don't have to manually enforce every rule:

- **`black`** — automatically reformats your code to a consistent style
- **`ruff`** (or `flake8`) — lints your code and flags style/quality violations
- **`isort`** — automatically sorts and groups your imports

We'll go through PEP8's actual rules in detail in Part B.


```bash
pip install black ruff isort
black my_script.py
ruff check my_script.py
isort my_script.py
```


These three tools are commonly wired into **pre-commit hooks** and **CI/CD pipelines** — meaning a pull request can be automatically blocked from merging if the formatting or lint checks fail, before a human reviewer even looks at it.


## A13. Document Your Functions (Docstrings)

A **docstring** is a string placed right under a function definition that documents what it does, what parameters it takes, and what it returns. It shows up automatically in IDE tooltips and can be used to auto-generate documentation websites.


In [12]:
def calculate_tax(amount, rate=0.18):
    """
    Calculate tax on a given amount.

    Args:
        amount (float): base amount
        rate (float): tax rate, default 0.18

    Returns:
        float: tax amount
    """
    return amount * rate

print(calculate_tax(1000))
print(calculate_tax.__doc__)


180.0

    Calculate tax on a given amount.

    Args:
        amount (float): base amount
        rate (float): tax rate, default 0.18

    Returns:
        float: tax amount
    


Documentation tools like **Sphinx** and **MkDocs** can generate a complete, browsable API reference straight from your docstrings — no separate documentation-writing effort needed if your docstrings are good.


---
# Part B — What is PEP8?


## B0. What is a PEP? What is PEP8?

**PEP** stands for **Python Enhancement Proposal** — an official design document that proposes a new feature, process, or convention for the Python language. 

- **PEP 8** — the official Python **style guide** (what this section is about)


**PEP8 specifically** is a style guide: a set of conventions for how Python code should look — indentation, naming, spacing, line length, and more.


## B1. Why Was PEP8 Created?

Imagine three developers on the same team, each writing the exact same logic but in their own personal style:


In [13]:
# Developer 1
def calcTotal(a,b):return a+b

# Developer 2
def calc_total(a, b):
    return a + b

# Developer 3
def CalcTotal( a , b ):
    return a+b


## B3. Rule 1 — Indentation (4 spaces)


In [15]:
# Good
def greet():
    print("Hello")


```python
# Bad (inconsistent — mixing tabs and spaces, or using fewer/more than 4 spaces)
def greet():
  print("Hello")
```


## B4. Rule 2 — Maximum Line Length (79–99 characters)

Long lines are hard to read, hard to review in a side-by-side diff, and often force horizontal scrolling.


In [16]:
base_price, tax_rate, service_fee_rate, shipping_cost = 100, 0.18, 0.05, 10

# Bad — one long unreadable line
total_price = base_price + (base_price * tax_rate) + (base_price * service_fee_rate) + shipping_cost
print(total_price)

# Good — broken across lines
total_price = (
    base_price
    + base_price * tax_rate
    + base_price * service_fee_rate
    + shipping_cost
)
print(total_price)


133.0
133.0


## B5. Rule 3 — Function Names → `snake_case`


In [17]:
# Bad
def CalculateTotal():
    pass

# Good
def calculate_total():
    pass


## B6. Rule 4 — Variable Names → `snake_case`


In [18]:
# Bad
userName = "John"

# Good
user_name = "John"
print(user_name)


John


## B7. Rule 5 — Class Names → `PascalCase`

Classes are named differently from functions/variables specifically so you can tell at a glance whether an identifier refers to a class or not.


In [19]:
# Bad
class employee_record:
    pass

# Good
class EmployeeRecord:
    pass


## B8. Rule 6 — Constants → `UPPER_CASE`


In [20]:
max_retries = 3        # bad — looks like a regular variable
MAX_RETRIES = 3         # good — instantly signals "this doesn't change"


## B9. Rule 7 — Imports at the Top, Grouped

PEP8 recommends grouping imports in this order: **standard library** → **third-party packages** → **local application imports**, with a blank line between each group.


```python
import os
import sys

import requests
import pandas as pd

from myapp.utils import helper
```


## B10. Rule 8 — Spaces Around Operators


In [21]:
x=1+2          # bad
x = 1 + 2      # good
print(x)


3


## B11. Rule 9 — Blank Lines

PEP8's convention: **2 blank lines** between top-level functions/classes, and **1 blank line** between methods inside a class. This gives the eye a visual break between logically separate pieces of code.


## B12. Rule 10 — Comments (Meaningful, Not Redundant)

A comment should explain **why**, not restate **what** the code obviously already says.


In [22]:
x = 5

x = x + 1  # increment x     <- bad, this just restates the code

# Increment retry counter before the next attempt   <- good, explains WHY
x = x + 1
print(x)


7


## B13. Before / After — A Real Example: the `Employee` Class

**Before** (violates almost every PEP8 rule):


In [23]:
# BEFORE (violates PEP8)
class employee:
    def __init__(self,Name,sal):
        self.Name=Name
        self.sal=sal
    def GetSalary(self):
        return self.sal


**After** (PEP8 compliant):


In [24]:
# AFTER (PEP8 compliant)
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def get_salary(self):
        return self.salary

e = Employee("Priya", 50000)
print(e.get_salary())


50000


This exact "before/after" pattern is what a real code review comment thread looks like — a reviewer pointing out naming, spacing, and casing issues on a pull request.


## B14. Tools That Enforce PEP8 Automatically

You don't need to manually check every rule yourself — these tools do it for you:

- **`black`** — an opinionated auto-formatter; it rewrites your code to comply, no debate needed
- **`ruff`** (or the older `flake8`) — a linter that flags violations without changing your code
- **`isort`** — automatically sorts and groups your imports
- Most IDEs (VS Code, PyCharm) can run these automatically every time you save a file


```bash
pip install black ruff isort
black .
ruff check .
isort .
```


These tools are typically wired into **pre-commit hooks** and **CI pipelines** at real companies — a pull request can fail the build automatically if formatting or lint checks don't pass, before any human even reviews the logic.
